
# Phase 1: alpha(T) extraction from log|V|-log|I| power-law fits

Outputs:
- `analysis_code/figures/phase_1/1a_alpha_vs_T.png`
- `analysis_code/figures/phase_1/1b_TBKT_comparison.png`
- `analysis_code/figures/phase_1/1c_residual_T{xx}K.png` (one per T)
- `analysis_code/results/phase1_alpha_table.csv`


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

# ── paths ──────────────────────────────────────────────────────────────────
def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / 'Processed_data').exists() and (p / 'analysis_code').exists():
            return p
    raise FileNotFoundError('Cannot find project root containing Processed_data/ and analysis_code/.')

ROOT = find_project_root()
CSV  = ROOT / 'Processed_data/analysis_inputs/all_cleaned_iv.csv'
FIG1 = ROOT / 'analysis_code/figures/phase_1'
RES  = ROOT / 'analysis_code/results'
FIG1.mkdir(parents=True, exist_ok=True)
RES.mkdir(parents=True, exist_ok=True)

# ── global style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 220,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

# ── known HN-fit T_BKT (from RT_Curve_analysis.ipynb, 316 nA excluded) ──
HN_TBKT = {
    'RT_I_10_uA':   86.138,
    'RT_I_3.16_uA': 85.643,
    'RT_I_1_uA':    83.983,
}
HN_TBKT_MEAN  = np.mean(list(HN_TBKT.values()))
HN_TBKT_STD   = np.std(list(HN_TBKT.values()))

# ── cut-off constants ─────────────────────────────────────────────────────
# Exclude low-current regime where log10(|I|) < -2 (likely measurement error)
LOGI_MIN = -2.0
I_MIN = 10**LOGI_MIN
V_MIN = 1e-7

print('ROOT:', ROOT)
print('CSV :', CSV)
print(f'Fit/plot cutoff: log10(|I|) >= {LOGI_MIN:.1f} (|I| >= {I_MIN:.2e} A)')


ROOT: /Users/pirin/Desktop/Lab_summer_research_BSCCO_device
CSV : /Users/pirin/Desktop/Lab_summer_research_BSCCO_device/Processed_data/analysis_inputs/all_cleaned_iv.csv
Fit/plot cutoff: log10(|I|) >= -2.0 (|I| >= 1.00e-02 A)


In [2]:

# ── load data ─────────────────────────────────────────────────────────────
df = pd.read_csv(CSV)

# parse temperature string "2K" -> 2.0
def parse_temp(s):
    return float(str(s).replace('K', '').strip())

df['T_K'] = df['temp'].apply(parse_temp)
df['absI'] = df['I'].abs()
df['absV'] = df['V'].abs()

temps_all = sorted(df['T_K'].unique())
print(f'Total temperatures: {len(temps_all)}')
print(f'T range: {temps_all[0]} – {temps_all[-1]} K')


Total temperatures: 27
T range: 2.0 – 110.0 K


In [3]:

# ── Phase 1.1: fit alpha(T) per temperature ───────────────────────────────
records = []

for T in temps_all:
    sub = df[(df['T_K'] == T) & (df['absI'] >= I_MIN) & (df['absV'] > V_MIN)].copy()

    if len(sub) < 8:
        print(f'  T={T:5.1f} K  skip (only {len(sub)} usable points)')
        continue

    x = np.log10(sub['absI'].values)
    y = np.log10(sub['absV'].values)

    # linear fit in log-log space: log|V| = alpha * log|I| + c
    slope, intercept, r, p, se = stats.linregress(x, y)
    residuals = y - (slope * x + intercept)
    dyn_range = x.max() - x.min()
    r2 = r**2

    # runs test on residual sign sequence
    signs  = np.sign(residuals - np.median(residuals))
    runs   = 1 + np.sum(signs[:-1] != signs[1:])
    n_pos  = np.sum(signs > 0)
    n_neg  = np.sum(signs <= 0)
    n_tot  = n_pos + n_neg
    runs_exp = 2 * n_pos * n_neg / n_tot + 1
    runs_var = (2*n_pos*n_neg*(2*n_pos*n_neg - n_tot)) / (n_tot**2 * (n_tot-1)) if n_tot > 1 else np.nan
    z_runs   = (runs - runs_exp) / np.sqrt(runs_var) if (runs_var and runs_var > 0) else np.nan
    p_runs   = 2*(1 - stats.norm.cdf(abs(z_runs))) if np.isfinite(z_runs) else np.nan

    records.append({
        'T_K': T,
        'alpha': slope,
        'alpha_se': se,
        'intercept': intercept,
        'R2': r2,
        'dyn_range': dyn_range,
        'n_pts': len(sub),
        'z_runs': z_runs,
        'p_runs': p_runs,
        'residuals': residuals,
        'x_log': x,
        'y_log': y,
    })

    print(f'  T={T:5.1f} K  alpha={slope:6.3f} ±{se:.3f}  R²={r2:.4f}  dyn={dyn_range:.2f}  p_runs={p_runs:.3f}  n={len(sub)}')

print(f'\nFitted {len(records)} temperature nodes.')


  T=  2.0 K  alpha= 4.263 ±0.103  R²=0.6394  dyn=0.89  p_runs=0.000  n=967
  T=  3.0 K  alpha= 4.301 ±0.112  R²=0.6418  dyn=0.89  p_runs=0.000  n=822
  T=  4.0 K  alpha= 4.293 ±0.123  R²=0.6425  dyn=0.89  p_runs=0.000  n=685
  T=  5.0 K  alpha= 4.286 ±0.112  R²=0.6427  dyn=0.89  p_runs=0.000  n=821
  T=  8.0 K  alpha= 4.280 ±0.113  R²=0.6367  dyn=0.89  p_runs=0.000  n=821
  T= 10.0 K  alpha= 4.157 ±0.069  R²=0.6681  dyn=0.89  p_runs=0.000  n=1816
  T= 15.0 K  alpha= 4.351 ±0.112  R²=0.6475  dyn=0.89  p_runs=0.000  n=822
  T= 20.0 K  alpha= 4.442 ±0.110  R²=0.6632  dyn=0.89  p_runs=0.000  n=827
  T= 25.0 K  alpha= 4.391 ±0.100  R²=0.6684  dyn=0.89  p_runs=0.000  n=950
  T= 30.0 K  alpha= 3.164 ±0.112  R²=0.5666  dyn=1.00  p_runs=0.000  n=617
  T= 35.0 K  alpha= 3.104 ±0.075  R²=0.6379  dyn=1.00  p_runs=0.000  n=963
  T= 40.0 K  alpha= 2.710 ±0.022  R²=0.9310  dyn=1.00  p_runs=0.000  n=1092
  T= 45.0 K  alpha= 2.524 ±0.023  R²=0.9423  dyn=1.00  p_runs=0.000  n=720
  T= 50.0 K  alpha= 1.9

In [4]:

# ── Phase 1.3: sweep_part fitting plots (MXTMN vs MNTMX, one per T) ───────
# sweep_part definition:
#   MXTMN: current sweep from positive maximum to negative minimum.
#   MNTMX: current sweep from negative minimum to positive maximum.
SWEEP_LEFT = 'MXTMN'
SWEEP_RIGHT = 'MNTMX'

def _plot_sweep_fit(ax, sub, T, sweep_tag, data_color, fit_color):
    if len(sub) < 8:
        ax.text(0.5, 0.5, f'{sweep_tag}: insufficient points', transform=ax.transAxes,
                ha='center', va='center', fontsize=10)
        ax.set_xlabel(r'$\log_{10}|I|$')
        ax.set_ylabel(r'$\log_{10}|V|$')
        ax.set_title(f'{sweep_tag} | T = {T} K')
        return

    x = np.log10(sub['absI'].values)
    y = np.log10(sub['absV'].values)
    slope, intercept, r, p, se = stats.linregress(x, y)

    x_fit = np.linspace(x.min(), x.max(), 200)
    y_fit = slope * x_fit + intercept

    ax.scatter(x, y, s=10, alpha=0.6, color=data_color, label='data')
    ax.plot(x_fit, y_fit, lw=1.8, color=fit_color,
            label=f"α = {slope:.3f} ± {se:.3f}  (n={len(sub)})")
    ax.set_xlabel(r'$\log_{10}|I|$')
    ax.set_ylabel(r'$\log_{10}|V|$')
    ax.set_title(f'{sweep_tag} | T = {T} K')
    ax.legend(fontsize=8)

for rec in records:
    T = rec['T_K']
    sub_mxtmn = df[(df['T_K'] == T) & (df['sweep_part'] == SWEEP_LEFT) & (df['absI'] >= I_MIN) & (df['absV'] > V_MIN)]
    sub_mntmx = df[(df['T_K'] == T) & (df['sweep_part'] == SWEEP_RIGHT) & (df['absI'] >= I_MIN) & (df['absV'] > V_MIN)]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    _plot_sweep_fit(axes[0], sub_mxtmn, T, SWEEP_LEFT, 'steelblue', 'tomato')
    _plot_sweep_fit(axes[1], sub_mntmx, T, SWEEP_RIGHT, 'darkorange', 'purple')

    plt.tight_layout()
    # Keep filename for report compatibility.
    fname = FIG1 / f"1c_residual_T{int(T):03d}K.png"
    plt.savefig(fname, dpi=220)
    plt.close()

# save results table
alpha_df = pd.DataFrame([
    {k: v for k, v in rec.items() if k not in ('residuals', 'x_log', 'y_log')}
    for rec in records
])
alpha_df.to_csv(RES / 'phase1_alpha_table.csv', index=False)
print(f"\nSaved: {RES}/phase1_alpha_table.csv")



Saved: /Users/pirin/Desktop/Lab_summer_research_BSCCO_device/analysis_code/results/phase1_alpha_table.csv


In [5]:

# ── Phase 1.2a: alpha(T) vs T ─────────────────────────────────────────────
T_vals   = alpha_df['T_K'].values
alpha_v  = alpha_df['alpha'].values
alpha_se = alpha_df['alpha_se'].values
p_runs_vals = alpha_df['p_runs'].values

cmap_pts = ['tomato' if (np.isfinite(p) and p < 0.05) else 'steelblue' for p in p_runs_vals]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(T_vals, alpha_v, yerr=2*alpha_se, fmt='none', ecolor='gray', elinewidth=1, capsize=3, zorder=1)
ax.scatter(T_vals, alpha_v, c=cmap_pts, s=55, zorder=3,
           label='blue = random residual, red = structured residual (p<0.05)')
ax.axhline(3, color='crimson', ls='--', lw=1.5, label=r'$\alpha = 3$ (BKT criterion)')
ax.axhline(1, color='navy', ls=':', lw=1.2, label=r'$\alpha = 1$ (Ohmic)')
ax.axvspan(HN_TBKT_MEAN - HN_TBKT_STD, HN_TBKT_MEAN + HN_TBKT_STD,
           alpha=0.15, color='gold', label=f"HN fit $T_{{BKT}}$ = {HN_TBKT_MEAN:.1f}±{HN_TBKT_STD:.1f} K")
ax.set_xlabel('Temperature (K)')
ax.set_ylabel(r'Power-law exponent $\alpha$')
ax.set_title(r'Phase 1a: $\alpha(T)$ from $\log|V|$–$\log|I|$ fits')
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(left=0)
plt.tight_layout()
fig.savefig(FIG1 / '1a_alpha_vs_T.png', dpi=220)
plt.close()
print(f'Saved: {FIG1}/1a_alpha_vs_T.png')

# ── Phase 1.2b: T_BKT cross-check ─────────────────────────────────────────
valid_mask = np.isfinite(alpha_v)
T_v2   = T_vals[valid_mask]
alp_v2 = alpha_v[valid_mask]

idx_s  = np.argsort(T_v2)
T_v2   = T_v2[idx_s]
alp_v2 = alp_v2[idx_s]

crossings = []
for i in range(len(alp_v2) - 1):
    if (alp_v2[i] >= 3 and alp_v2[i+1] < 3) or (alp_v2[i] <= 3 and alp_v2[i+1] > 3):
        T_cross = T_v2[i] + (3 - alp_v2[i]) * (T_v2[i+1] - T_v2[i]) / (alp_v2[i+1] - alp_v2[i])
        crossings.append(T_cross)

print(f'\nalpha=3 crossing(s): {crossings}')
T_BKT_alpha = crossings[-1] if crossings else np.nan

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(T_v2, alp_v2, 'o-', color='steelblue', ms=5, lw=1.4, label=r'$\alpha(T)$ from IV fits')
ax.axhline(3, color='crimson', ls='--', lw=1.5, label=r'$\alpha = 3$')
ax.axvspan(min(HN_TBKT.values()), max(HN_TBKT.values()), alpha=0.2, color='gold',
           label=f"HN fit $T_{{BKT}}$ range ({min(HN_TBKT.values()):.1f}–{max(HN_TBKT.values()):.1f} K)")

if np.isfinite(T_BKT_alpha):
    ax.axvline(T_BKT_alpha, color='forestgreen', ls='-.', lw=1.5,
               label=f"$T_{{BKT}}$ from α=3 crossing: {T_BKT_alpha:.1f} K")
    diff = abs(T_BKT_alpha - HN_TBKT_MEAN)
    note = f"Δ = {diff:.1f} K" + (" ← finite-size/dynamic crossover?" if diff > 1 else "")
    ax.text(T_BKT_alpha + 0.3, 3.3, note, color='forestgreen', fontsize=9)

ax.set_xlabel('Temperature (K)')
ax.set_ylabel(r'$\alpha(T)$')
ax.set_title(r'Phase 1b: $T_{BKT}$ cross-check: $\alpha=3$ vs HN tail fit')
ax.set_xlim(75, 100)
ax.set_ylim(bottom=0)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(FIG1 / '1b_TBKT_comparison.png', dpi=220)
plt.close()
print(f'Saved: {FIG1}/1b_TBKT_comparison.png')

# console summary
print('\n' + '='*60)
print('PHASE 1 SUMMARY')
print('='*60)
if np.any(alp_v2 > 3):
    print(f"T range with alpha > 3 : {T_v2[alp_v2 > 3].min():.1f} – {T_v2[alp_v2 > 3].max():.1f} K")
else:
    print('T range with alpha > 3 : none')
print(f"T_BKT from alpha=3 crossing : {T_BKT_alpha:.2f} K")
print(f"T_BKT from HN tail fit (mean): {HN_TBKT_MEAN:.2f} K")
print(f"Discrepancy : {abs(T_BKT_alpha - HN_TBKT_MEAN):.2f} K")
print()
structured = alpha_df[alpha_df['p_runs'] < 0.05]
print('Temperatures with structured log-log residuals (p_runs < 0.05): '
      f"{sorted(structured['T_K'].tolist())}")
print()
print('Model suitability hint:')
print('  Pure power-law (Model A) likely OK at T where p_runs > 0.05')
print('  LAMH / RSJ needed where residual is structured (p_runs < 0.05)')
print('='*60)


Saved: /Users/pirin/Desktop/Lab_summer_research_BSCCO_device/analysis_code/figures/phase_1/1a_alpha_vs_T.png

alpha=3 crossing(s): [np.float64(36.31516669813897), np.float64(79.10114064496632), np.float64(85.16579156382677)]


Saved: /Users/pirin/Desktop/Lab_summer_research_BSCCO_device/analysis_code/figures/phase_1/1b_TBKT_comparison.png

PHASE 1 SUMMARY
T range with alpha > 3 : 2.0 – 85.0 K
T_BKT from alpha=3 crossing : 85.17 K
T_BKT from HN tail fit (mean): 85.25 K
Discrepancy : 0.09 K

Temperatures with structured log-log residuals (p_runs < 0.05): [2.0, 3.0, 4.0, 5.0, 8.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0, 45.0, 50.0, 55.0, 60.0, 65.0, 68.0, 70.0, 75.0, 80.0, 85.0, 88.0, 90.0, 95.0, 100.0, 110.0]

Model suitability hint:
  Pure power-law (Model A) likely OK at T where p_runs > 0.05
  LAMH / RSJ needed where residual is structured (p_runs < 0.05)
